# Chapter 9.1: Large-Scale Data Pipelines for Recommendation Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement** multiple data sampling strategies (uniform, popularity-weighted, hard-negative mining) for recommendation training
2. **Design** negative sampling pipelines using in-batch, cached, and mixed strategies at scale
3. **Compare** data formats (TFRecord, Parquet, Petastorm) and their trade-offs for rec training
4. **Build** streaming data generation pipelines for real-time training
5. **Apply** data quality techniques including deduplication, spam filtering, and bot detection
6. **Evaluate** the impact of different sampling strategies on model convergence and quality
7. **Construct** an end-to-end negative sampling pipeline combining multiple strategies

## Prerequisites

- Python 3.8+, NumPy, PyTorch basics
- Understanding of recommendation models (matrix factorization, two-tower)
- Familiarity with implicit feedback datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part9/chapter_9.1_data_pipelines.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/blob/main/notebooks/part9/chapter_9.1_data_pipelines.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import time
import json
import hashlib
import random
from typing import List, Dict, Tuple, Optional

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")

## 1. Data Sampling Strategies for Recommendation Systems

In recommendation systems, the distribution of interactions follows a heavy power-law: a few popular items account for most interactions, while the long tail of items has very sparse feedback. Sampling strategy directly impacts model quality.

### Three Core Strategies

**Uniform Sampling:** Each item has equal probability of being sampled as a negative.

$$P_{\text{uniform}}(i) = \frac{1}{|\mathcal{I}|}$$

**Popularity-Weighted Sampling** (Mikolov et al., 2013 — Google): Items are sampled proportional to their frequency raised to a power $\alpha$:

$$P_{\text{pop}}(i) = \frac{f(i)^\alpha}{\sum_{j \in \mathcal{I}} f(j)^\alpha}$$

where $\alpha = 0.75$ is commonly used (borrowed from Word2Vec).

**Hard-Negative Mining** (Rendle et al., 2020 — Google): Negatives are sampled based on model scores, focusing on items the model incorrectly ranks highly:

$$P_{\text{hard}}(i | u) \propto \exp(s(u, i) / \tau)$$

where $s(u, i)$ is the model's predicted score and $\tau$ is a temperature parameter.

> **💡 Concept:** The choice of negative sampling distribution is one of the most impactful decisions in training recommendation models with implicit feedback. Uniform sampling is simple but can lead to trivially easy negatives; hard-negative mining accelerates convergence but risks training instability.

In [ ]:
# Generate synthetic interaction data following a power-law distribution
NUM_USERS = 10000
NUM_ITEMS = 50000
NUM_INTERACTIONS = 500000

# Item popularity follows Zipf's law
item_popularity = np.random.zipf(a=1.5, size=NUM_ITEMS).astype(np.float64)
item_popularity = np.clip(item_popularity, 1, 1000)
item_popularity /= item_popularity.sum()

# Generate interactions
user_ids = np.random.randint(0, NUM_USERS, size=NUM_INTERACTIONS)
item_ids = np.random.choice(NUM_ITEMS, size=NUM_INTERACTIONS, p=item_popularity)

# Build user-item interaction sets
user_items = defaultdict(set)
for u, i in zip(user_ids, item_ids):
    user_items[u].add(i)

# Item frequency counts
item_counts = Counter(item_ids)

print(f"Generated {NUM_INTERACTIONS} interactions")
print(f"Top-10 items account for {sum(v for _, v in item_counts.most_common(10)) / NUM_INTERACTIONS:.1%} of interactions")
print(f"Items with <5 interactions: {sum(1 for c in item_counts.values() if c < 5)} / {NUM_ITEMS}")

In [ ]:
class NegativeSampler:
    """Implements multiple negative sampling strategies for recommendation."""
    
    def __init__(self, num_items: int, item_counts: Dict[int, int], alpha: float = 0.75):
        self.num_items = num_items
        self.item_counts = item_counts
        self.alpha = alpha
        
        # Precompute popularity-weighted distribution
        counts = np.zeros(num_items, dtype=np.float64)
        for item_id, count in item_counts.items():
            if item_id < num_items:
                counts[item_id] = count
        counts = np.maximum(counts, 1.0)  # Laplace smoothing
        self.pop_probs = counts ** alpha
        self.pop_probs /= self.pop_probs.sum()
    
    def uniform_sample(self, user_positives: set, num_negatives: int) -> np.ndarray:
        """Uniform random negative sampling."""
        negatives = []
        while len(negatives) < num_negatives:
            candidates = np.random.randint(0, self.num_items, size=num_negatives * 2)
            for c in candidates:
                if c not in user_positives:
                    negatives.append(c)
                    if len(negatives) >= num_negatives:
                        break
        return np.array(negatives[:num_negatives])
    
    def popularity_sample(self, user_positives: set, num_negatives: int) -> np.ndarray:
        """Popularity-weighted negative sampling (Word2Vec style)."""
        negatives = []
        while len(negatives) < num_negatives:
            candidates = np.random.choice(self.num_items, size=num_negatives * 2, p=self.pop_probs)
            for c in candidates:
                if c not in user_positives:
                    negatives.append(c)
                    if len(negatives) >= num_negatives:
                        break
        return np.array(negatives[:num_negatives])
    
    def hard_negative_sample(self, user_positives: set, model_scores: np.ndarray,
                              num_negatives: int, temperature: float = 1.0) -> np.ndarray:
        """Hard-negative mining based on model scores."""
        # Mask out positive items
        scores = model_scores.copy()
        for p in user_positives:
            if p < len(scores):
                scores[p] = -np.inf
        
        # Softmax with temperature
        scores = scores / temperature
        scores = scores - scores.max()
        exp_scores = np.exp(scores)
        probs = exp_scores / exp_scores.sum()
        
        negatives = np.random.choice(self.num_items, size=num_negatives, p=probs, replace=False)
        return negatives


# Demonstrate
sampler = NegativeSampler(NUM_ITEMS, item_counts)
test_user_positives = user_items[0]
print(f"User 0 has {len(test_user_positives)} positive items")

uniform_negs = sampler.uniform_sample(test_user_positives, 10)
pop_negs = sampler.popularity_sample(test_user_positives, 10)
print(f"Uniform negatives (first 10): {uniform_negs}")
print(f"Popularity negatives (first 10): {pop_negs}")

In [ ]:
# Visualize the sampling distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

n_samples = 100000
test_positives = set(range(10))  # small positive set for illustration

# Uniform
uniform_samples = np.random.randint(0, NUM_ITEMS, size=n_samples)
axes[0].hist(uniform_samples, bins=100, density=True, alpha=0.7, color='steelblue')
axes[0].set_title('Uniform Sampling', fontsize=13)
axes[0].set_xlabel('Item ID')
axes[0].set_ylabel('Density')

# Popularity-weighted
pop_samples = np.random.choice(NUM_ITEMS, size=n_samples, p=sampler.pop_probs)
axes[1].hist(pop_samples, bins=100, density=True, alpha=0.7, color='coral')
axes[1].set_title('Popularity-Weighted Sampling (α=0.75)', fontsize=13)
axes[1].set_xlabel('Item ID')
axes[1].set_ylabel('Density')

# Show popularity distribution
sorted_counts = sorted(item_counts.values(), reverse=True)
axes[2].loglog(range(1, len(sorted_counts) + 1), sorted_counts, alpha=0.7, color='green')
axes[2].set_title('Item Popularity (log-log)', fontsize=13)
axes[2].set_xlabel('Item Rank')
axes[2].set_ylabel('Interaction Count')

plt.tight_layout()
plt.show()

## 2. Negative Sampling at Scale

At production scale (YouTube, TikTok, Meta), negative sampling must be both **efficient** and **effective**. Three main approaches have emerged:

### In-Batch Negatives (Yi et al., 2019 — Google)
Use other items in the same mini-batch as negatives. This is extremely efficient — no extra computation needed — but introduces **sampling bias** proportional to item popularity.

$$\mathcal{L}_{\text{in-batch}} = -\log \frac{\exp(s(u, i^+))}{\exp(s(u, i^+)) + \sum_{j \in \mathcal{B} \setminus \{i^+\}} \exp(s(u, j))}$$

### Cached Negatives (Lindgren et al., 2021)
Maintain a FIFO queue of recent item embeddings. This provides a large pool of negatives cheaply, with slightly stale representations.

### Mixed Strategy
Combine easy negatives (uniform/popularity) with hard negatives (model-scored) to balance training stability and convergence speed.

> **⚠️ Common Pitfall:** Pure in-batch negative sampling introduces a popularity bias: popular items appear more frequently as negatives, making the model overly cautious about recommending them. Correction factors (log-Q correction from Yi et al., 2019) are essential.

In [ ]:
class InBatchNegativeSampling:
    """In-batch negative sampling with optional log-Q correction."""
    
    def __init__(self, item_frequencies: np.ndarray, temperature: float = 0.1):
        self.temperature = temperature
        # Precompute log-Q correction (Yi et al., 2019)
        freq = item_frequencies / item_frequencies.sum()
        self.log_q = torch.tensor(np.log(freq + 1e-10), dtype=torch.float32)
    
    def compute_loss(self, user_embs: torch.Tensor, item_embs: torch.Tensor,
                     item_ids: torch.Tensor, apply_correction: bool = True) -> torch.Tensor:
        """
        Compute in-batch softmax loss with optional log-Q correction.
        user_embs: (B, D), item_embs: (B, D), item_ids: (B,)
        """
        # Compute all pairwise scores: (B, B)
        logits = torch.mm(user_embs, item_embs.t()) / self.temperature
        
        if apply_correction:
            # Subtract log-Q to correct for popularity bias
            correction = self.log_q[item_ids].unsqueeze(0)  # (1, B)
            logits = logits - correction
        
        # Labels: diagonal entries are positives
        labels = torch.arange(logits.size(0))
        loss = F.cross_entropy(logits, labels)
        return loss


class CachedNegativeSampling:
    """Maintains a FIFO cache of item embeddings for negative sampling."""
    
    def __init__(self, cache_size: int, embedding_dim: int):
        self.cache_size = cache_size
        self.cache = torch.zeros(cache_size, embedding_dim)
        self.cache_ids = torch.zeros(cache_size, dtype=torch.long)
        self.ptr = 0
        self.full = False
    
    def update(self, item_embs: torch.Tensor, item_ids: torch.Tensor):
        """Add new item embeddings to cache."""
        batch_size = item_embs.size(0)
        with torch.no_grad():
            if self.ptr + batch_size <= self.cache_size:
                self.cache[self.ptr:self.ptr + batch_size] = item_embs.detach()
                self.cache_ids[self.ptr:self.ptr + batch_size] = item_ids.detach()
            else:
                overflow = (self.ptr + batch_size) - self.cache_size
                remaining = batch_size - overflow
                self.cache[self.ptr:self.ptr + remaining] = item_embs[:remaining].detach()
                self.cache_ids[self.ptr:self.ptr + remaining] = item_ids[:remaining].detach()
                self.cache[:overflow] = item_embs[remaining:].detach()
                self.cache_ids[:overflow] = item_ids[remaining:].detach()
                self.full = True
            self.ptr = (self.ptr + batch_size) % self.cache_size
    
    def get_negatives(self, num_negatives: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample negatives from cache."""
        max_idx = self.cache_size if self.full else self.ptr
        if max_idx == 0:
            return torch.zeros(num_negatives, self.cache.size(1)), torch.zeros(num_negatives, dtype=torch.long)
        indices = torch.randint(0, max_idx, (num_negatives,))
        return self.cache[indices], self.cache_ids[indices]


# Demo: In-batch negatives
EMB_DIM = 64
BATCH_SIZE = 256

freq_array = np.zeros(NUM_ITEMS)
for item_id, count in item_counts.items():
    if item_id < NUM_ITEMS:
        freq_array[item_id] = count
freq_array = np.maximum(freq_array, 1.0)

in_batch_sampler = InBatchNegativeSampling(freq_array)

# Simulate a batch
user_embs = F.normalize(torch.randn(BATCH_SIZE, EMB_DIM), dim=1)
item_embs = F.normalize(torch.randn(BATCH_SIZE, EMB_DIM), dim=1)
item_ids_batch = torch.from_numpy(np.random.choice(NUM_ITEMS, size=BATCH_SIZE, p=item_popularity))

loss_no_corr = in_batch_sampler.compute_loss(user_embs, item_embs, item_ids_batch, apply_correction=False)
loss_with_corr = in_batch_sampler.compute_loss(user_embs, item_embs, item_ids_batch, apply_correction=True)

print(f"In-batch loss (no correction): {loss_no_corr.item():.4f}")
print(f"In-batch loss (log-Q correction): {loss_with_corr.item():.4f}")

In [ ]:
# Demo: Cached negatives
cache = CachedNegativeSampling(cache_size=10000, embedding_dim=EMB_DIM)

# Simulate filling cache over multiple batches
for _ in range(20):
    batch_embs = F.normalize(torch.randn(BATCH_SIZE, EMB_DIM), dim=1)
    batch_ids = torch.randint(0, NUM_ITEMS, (BATCH_SIZE,))
    cache.update(batch_embs, batch_ids)

neg_embs, neg_ids = cache.get_negatives(512)
print(f"Cache status: {'full' if cache.full else 'partially filled'}")
print(f"Sampled {neg_embs.shape[0]} cached negatives with dim={neg_embs.shape[1]}")
print(f"Unique items in cached negatives: {neg_ids.unique().shape[0]}")

## 3. Data Formats for Recommendation Training

At scale, the choice of data format impacts I/O throughput, storage cost, and training speed.

| Format | Strengths | Weaknesses | Best For |
|--------|-----------|------------|----------|
| **TFRecord** | Streaming-friendly, TF-native | Not human-readable, TF-only | TensorFlow pipelines |
| **Parquet** | Columnar, compressed, cross-platform | Not optimized for streaming | Feature engineering, Spark |
| **Petastorm** | Parquet + PyTorch/TF DataLoader | Extra dependency | Multi-framework shops |
| **Arrow/IPC** | Zero-copy reads, very fast | Large file sizes | In-memory processing |

> **🔑 Pro Tip:** At companies like Meta and Google, custom binary formats are often used for maximum throughput. However, Parquet with snappy compression offers an excellent balance of speed and compatibility for most teams.

In [ ]:
# Simulate different data format serialization/deserialization performance
import struct
import io

class DataFormatBenchmark:
    """Benchmark simulated data format operations."""
    
    def __init__(self, num_samples: int, num_features: int):
        self.num_samples = num_samples
        self.num_features = num_features
        # Synthetic feature data
        self.user_ids = np.random.randint(0, NUM_USERS, size=num_samples)
        self.item_ids = np.random.randint(0, NUM_ITEMS, size=num_samples)
        self.features = np.random.randn(num_samples, num_features).astype(np.float32)
        self.labels = np.random.randint(0, 2, size=num_samples).astype(np.float32)
    
    def benchmark_binary(self) -> Tuple[float, float, int]:
        """Simulate binary format (like TFRecord) write/read."""
        buf = io.BytesIO()
        # Write
        t0 = time.time()
        for i in range(self.num_samples):
            buf.write(struct.pack('i', self.user_ids[i]))
            buf.write(struct.pack('i', self.item_ids[i]))
            buf.write(self.features[i].tobytes())
            buf.write(struct.pack('f', self.labels[i]))
        write_time = time.time() - t0
        size = buf.tell()
        
        # Read
        buf.seek(0)
        t0 = time.time()
        record_size = 4 + 4 + self.num_features * 4 + 4
        for _ in range(self.num_samples):
            data = buf.read(record_size)
        read_time = time.time() - t0
        
        return write_time, read_time, size
    
    def benchmark_columnar(self) -> Tuple[float, float, int]:
        """Simulate columnar format (like Parquet) write/read."""
        # Write: pack each column separately
        t0 = time.time()
        columns = {
            'user_ids': self.user_ids.tobytes(),
            'item_ids': self.item_ids.tobytes(),
            'features': self.features.tobytes(),
            'labels': self.labels.tobytes()
        }
        total_size = sum(len(v) for v in columns.values())
        write_time = time.time() - t0
        
        # Read: can read individual columns
        t0 = time.time()
        _ = np.frombuffer(columns['user_ids'], dtype=np.int64)
        _ = np.frombuffer(columns['features'], dtype=np.float32).reshape(-1, self.num_features)
        read_time = time.time() - t0
        
        return write_time, read_time, total_size
    
    def benchmark_numpy(self) -> Tuple[float, float, int]:
        """Simulate NumPy .npz format write/read."""
        buf = io.BytesIO()
        t0 = time.time()
        np.savez_compressed(buf, user_ids=self.user_ids, item_ids=self.item_ids,
                           features=self.features, labels=self.labels)
        write_time = time.time() - t0
        size = buf.tell()
        
        buf.seek(0)
        t0 = time.time()
        data = np.load(buf)
        _ = data['user_ids']
        _ = data['features']
        read_time = time.time() - t0
        
        return write_time, read_time, size


bench = DataFormatBenchmark(num_samples=10000, num_features=32)

results = {}
results['Binary (TFRecord-like)'] = bench.benchmark_binary()
results['Columnar (Parquet-like)'] = bench.benchmark_columnar()
results['Compressed (NPZ)'] = bench.benchmark_numpy()

print(f"{'Format':<25} {'Write (ms)':<12} {'Read (ms)':<12} {'Size (KB)':<12}")
print("-" * 60)
for name, (wt, rt, sz) in results.items():
    print(f"{name:<25} {wt*1000:<12.2f} {rt*1000:<12.2f} {sz/1024:<12.1f}")

## 4. Streaming Data: Real-Time Training Data Generation

Modern recommendation systems increasingly adopt **online learning** paradigms where training data is generated in real-time from user interactions.

Key challenges:
- **Ordering**: events arrive out of order across services
- **Feature freshness**: features must reflect the state at impression time, not training time
- **Label delay**: conversions (purchases) may happen hours/days after impressions
- **Throughput**: systems like Meta's process billions of events per day

The typical architecture uses a **feature logging pipeline** that joins impression-time features with delayed labels:

$$\text{Training Sample} = \text{Join}(\text{Impression}_{t}, \text{Label}_{t+\Delta}) \bowtie \text{Features}_{t}$$

> **⚠️ Common Pitfall:** A common data leakage bug in streaming pipelines is using features computed *after* the impression time. For example, using today's item click count to predict yesterday's click is a form of future information leakage.

In [ ]:
class StreamingDataGenerator:
    """Simulates a real-time training data stream for recommendation."""
    
    def __init__(self, num_users: int, num_items: int, item_pop: np.ndarray):
        self.num_users = num_users
        self.num_items = num_items
        self.item_pop = item_pop
        self.current_time = 0
        # Simulate feature drift: item features change over time
        self.item_embeddings = np.random.randn(num_items, 16).astype(np.float32)
        self.user_embeddings = np.random.randn(num_users, 16).astype(np.float32)
    
    def generate_batch(self, batch_size: int, label_delay: int = 0) -> Dict:
        """Generate a streaming batch of training data."""
        self.current_time += 1
        
        # Simulate feature drift (small random walk)
        drift = np.random.randn(self.num_items, 16).astype(np.float32) * 0.001
        self.item_embeddings += drift
        
        # Generate impressions
        users = np.random.randint(0, self.num_users, size=batch_size)
        items = np.random.choice(self.num_items, size=batch_size, p=self.item_pop)
        
        # Generate labels: probability depends on user-item similarity
        user_feats = self.user_embeddings[users]
        item_feats = self.item_embeddings[items]
        scores = np.sum(user_feats * item_feats, axis=1)
        click_probs = 1 / (1 + np.exp(-scores * 0.1))
        labels = (np.random.random(batch_size) < click_probs).astype(np.float32)
        
        return {
            'user_ids': users,
            'item_ids': items,
            'user_features': user_feats,
            'item_features': item_feats,
            'labels': labels,
            'timestamp': self.current_time,
            'label_delay': label_delay
        }


stream = StreamingDataGenerator(NUM_USERS, NUM_ITEMS, item_popularity)

# Simulate 50 streaming batches and track statistics
ctr_over_time = []
batch_sizes = []

for step in range(50):
    batch = stream.generate_batch(batch_size=1024)
    ctr = batch['labels'].mean()
    ctr_over_time.append(ctr)
    batch_sizes.append(len(batch['labels']))

print(f"Generated {sum(batch_sizes)} streaming samples over {len(ctr_over_time)} batches")
print(f"Average CTR: {np.mean(ctr_over_time):.4f} (±{np.std(ctr_over_time):.4f})")

## 5. Data Quality: Deduplication, Spam Filtering, Bot Detection

Data quality is critical for recommendation models. Noisy labels from bots, spam interactions, or duplicate events can severely degrade model performance.

### Key Techniques

1. **Deduplication**: Remove duplicate interactions (same user-item-timestamp within a window)
2. **Bot Detection**: Identify non-human interaction patterns (too-fast clicks, uniform distributions)
3. **Spam Filtering**: Remove interactions with known spam items or from flagged accounts

Statistical signals for bot detection:
- **Click rate**: Bots often have abnormally high CTR or uniform CTR across all items
- **Inter-arrival time**: Human clicks follow log-normal distribution; bots are often periodic
- **Diversity**: Real users have skewed preferences; bots may interact uniformly

In [ ]:
class DataQualityPipeline:
    """Data quality pipeline for recommendation training data."""
    
    def __init__(self, dedup_window: float = 1.0, min_inter_arrival: float = 0.1,
                 max_hourly_rate: int = 200, min_diversity: float = 0.1):
        self.dedup_window = dedup_window
        self.min_inter_arrival = min_inter_arrival
        self.max_hourly_rate = max_hourly_rate
        self.min_diversity = min_diversity
        self.seen_hashes = set()
    
    def deduplicate(self, interactions: List[Dict]) -> List[Dict]:
        """Remove duplicate interactions based on content hash."""
        unique = []
        for interaction in interactions:
            key = f"{interaction['user_id']}_{interaction['item_id']}_{interaction['timestamp']:.0f}"
            h = hashlib.md5(key.encode()).hexdigest()
            if h not in self.seen_hashes:
                self.seen_hashes.add(h)
                unique.append(interaction)
        return unique
    
    def detect_bots(self, user_interactions: Dict[int, List[Dict]]) -> set:
        """Detect bot users based on interaction patterns."""
        bot_users = set()
        for user_id, interactions in user_interactions.items():
            if len(interactions) < 5:
                continue
            
            # Check 1: Rate too high
            timestamps = sorted([x['timestamp'] for x in interactions])
            time_span = timestamps[-1] - timestamps[0]
            if time_span > 0:
                hourly_rate = len(interactions) / (time_span / 3600)
                if hourly_rate > self.max_hourly_rate:
                    bot_users.add(user_id)
                    continue
            
            # Check 2: Inter-arrival times too regular
            if len(timestamps) > 2:
                inter_arrivals = np.diff(timestamps)
                cv = np.std(inter_arrivals) / (np.mean(inter_arrivals) + 1e-10)
                if cv < 0.05:  # Too regular = likely bot
                    bot_users.add(user_id)
                    continue
            
            # Check 3: Item diversity too uniform
            items = [x['item_id'] for x in interactions]
            unique_ratio = len(set(items)) / len(items)
            if unique_ratio > 0.99 and len(items) > 50:  # Nearly all unique = suspicious
                bot_users.add(user_id)
        
        return bot_users


# Generate synthetic data with some bots and duplicates
np.random.seed(42)
interactions = []

# Normal users
for _ in range(10000):
    interactions.append({
        'user_id': np.random.randint(0, 900),
        'item_id': np.random.choice(1000, p=np.random.dirichlet(np.ones(1000) * 0.1)),
        'timestamp': np.random.uniform(0, 3600)
    })

# Bot users (very regular, high rate)
for bot_id in range(900, 910):
    for i in range(300):
        interactions.append({
            'user_id': bot_id,
            'item_id': np.random.randint(0, 1000),
            'timestamp': i * 1.0  # Perfectly regular intervals
        })

# Duplicate some interactions
duplicates = random.sample(interactions[:10000], 500)
interactions.extend(duplicates)

print(f"Total interactions (with duplicates and bots): {len(interactions)}")

pipeline = DataQualityPipeline()

# Deduplicate
clean = pipeline.deduplicate(interactions)
print(f"After deduplication: {len(clean)} (removed {len(interactions) - len(clean)} duplicates)")

# Detect bots
user_grouped = defaultdict(list)
for inter in clean:
    user_grouped[inter['user_id']].append(inter)

bots = pipeline.detect_bots(user_grouped)
print(f"Detected {len(bots)} bot users")
print(f"True bot IDs (900-909): {sorted([b for b in bots if b >= 900])}")

# Remove bot interactions
final = [x for x in clean if x['user_id'] not in bots]
print(f"Final clean dataset: {len(final)} interactions")

In [ ]:
# Visualize bot vs normal user patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inter-arrival time distribution
normal_user = 0
normal_times = sorted([x['timestamp'] for x in user_grouped[normal_user]])
normal_iat = np.diff(normal_times) if len(normal_times) > 1 else [0]

bot_user = 900
bot_times = sorted([x['timestamp'] for x in user_grouped[bot_user]])
bot_iat = np.diff(bot_times) if len(bot_times) > 1 else [0]

axes[0].hist(normal_iat, bins=30, alpha=0.7, label='Normal User', color='steelblue', density=True)
axes[0].hist(bot_iat, bins=30, alpha=0.7, label='Bot User', color='red', density=True)
axes[0].set_title('Inter-Arrival Time Distribution', fontsize=13)
axes[0].set_xlabel('Time between actions (s)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Interaction rate per user
user_rates = []
user_types = []
for uid, inters in user_grouped.items():
    if len(inters) >= 5:
        ts = sorted([x['timestamp'] for x in inters])
        span = ts[-1] - ts[0]
        rate = len(inters) / (span / 3600 + 0.01)
        user_rates.append(min(rate, 2000))
        user_types.append('Bot' if uid in bots else 'Normal')

normal_rates = [r for r, t in zip(user_rates, user_types) if t == 'Normal']
bot_rates = [r for r, t in zip(user_rates, user_types) if t == 'Bot']

axes[1].hist(normal_rates, bins=30, alpha=0.7, label=f'Normal (n={len(normal_rates)})', color='steelblue', density=True)
axes[1].hist(bot_rates, bins=30, alpha=0.7, label=f'Bot (n={len(bot_rates)})', color='red', density=True)
axes[1].set_title('Hourly Interaction Rate', fontsize=13)
axes[1].set_xlabel('Interactions per hour')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Impact of Sampling Strategy on Training

Let us now compare how different sampling strategies affect model training convergence and final quality. We will train a simple two-tower model with different negative sampling approaches.

> **💡 Concept:** The negative sampling strategy creates an implicit loss function modification. Popularity-weighted sampling approximates the NCE (Noise Contrastive Estimation) objective, while in-batch sampling approximates sampled softmax with a batch-dependent normalization constant.

In [ ]:
class SimpleTwoTower(nn.Module):
    """Simple two-tower model for comparing sampling strategies."""
    def __init__(self, num_users, num_items, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    
    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)
        v = self.item_emb(item_ids)
        return (u * v).sum(dim=-1)


def train_with_strategy(strategy: str, num_epochs: int = 10, batch_size: int = 512,
                        n_users: int = 1000, n_items: int = 5000, n_interactions: int = 50000):
    """Train a model with a given negative sampling strategy and return loss curves."""
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Generate synthetic data
    pop = np.random.zipf(a=1.5, size=n_items).astype(np.float64)
    pop = np.clip(pop, 1, 500)
    pop /= pop.sum()
    
    train_users = np.random.randint(0, n_users, size=n_interactions)
    train_items = np.random.choice(n_items, size=n_interactions, p=pop)
    
    u_items = defaultdict(set)
    for u, i in zip(train_users, train_items):
        u_items[u].add(i)
    
    counts = Counter(train_items)
    neg_sampler = NegativeSampler(n_items, counts)
    
    model = SimpleTwoTower(n_users, n_items)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCEWithLogitsLoss()
    
    losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0
        n_batches = 0
        indices = np.random.permutation(n_interactions)
        
        for start in range(0, n_interactions - batch_size, batch_size):
            batch_idx = indices[start:start + batch_size]
            pos_users = torch.tensor(train_users[batch_idx], dtype=torch.long)
            pos_items = torch.tensor(train_items[batch_idx], dtype=torch.long)
            
            # Sample negatives based on strategy
            neg_items_list = []
            for u in train_users[batch_idx]:
                positives = u_items[u]
                if strategy == 'uniform':
                    neg = neg_sampler.uniform_sample(positives, 1)
                elif strategy == 'popularity':
                    neg = neg_sampler.popularity_sample(positives, 1)
                else:  # mixed
                    if np.random.random() < 0.5:
                        neg = neg_sampler.uniform_sample(positives, 1)
                    else:
                        neg = neg_sampler.popularity_sample(positives, 1)
                neg_items_list.append(neg[0])
            
            neg_items = torch.tensor(neg_items_list, dtype=torch.long)
            
            # Forward pass
            pos_scores = model(pos_users, pos_items)
            neg_scores = model(pos_users, neg_items)
            
            scores = torch.cat([pos_scores, neg_scores])
            labels = torch.cat([torch.ones(batch_size), torch.zeros(batch_size)])
            
            loss = criterion(scores, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        losses.append(epoch_loss / max(n_batches, 1))
    
    return losses


# Train with different strategies
print("Training with uniform sampling...")
uniform_losses = train_with_strategy('uniform')
print("Training with popularity-weighted sampling...")
pop_losses = train_with_strategy('popularity')
print("Training with mixed sampling...")
mixed_losses = train_with_strategy('mixed')

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(uniform_losses, 'o-', label='Uniform', color='steelblue', linewidth=2)
ax.plot(pop_losses, 's-', label='Popularity-Weighted', color='coral', linewidth=2)
ax.plot(mixed_losses, '^-', label='Mixed (50/50)', color='green', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Convergence by Negative Sampling Strategy', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🏋️ Exercise 1: Implement a Multi-Strategy Negative Sampling Pipeline

Build a `MultiStrategyNegativeSampler` that combines uniform, popularity-weighted, and hard-negative sampling with configurable ratios. The sampler should:

1. Accept a ratio configuration (e.g., `{'uniform': 0.3, 'popularity': 0.4, 'hard': 0.3}`)
2. Generate the specified number of negatives with the appropriate mix
3. Track sampling statistics (how many from each strategy)

In [ ]:
# 🏋️ Exercise 1: Multi-strategy negative sampling pipeline

class MultiStrategyNegativeSampler:
    def __init__(self, num_items: int, item_counts: Dict[int, int],
                 strategy_ratios: Dict[str, float], alpha: float = 0.75):
        """
        Args:
            num_items: Total number of items
            item_counts: Item frequency counts
            strategy_ratios: Dict mapping strategy name to ratio, e.g.
                {'uniform': 0.3, 'popularity': 0.4, 'hard': 0.3}
            alpha: Smoothing exponent for popularity sampling
        """
        self.num_items = num_items
        self.strategy_ratios = strategy_ratios
        self.stats = {k: 0 for k in strategy_ratios}
        
        # TODO: Initialize the base sampler and any needed state
        # Hint: reuse the NegativeSampler class from above
        pass
    
    def sample(self, user_positives: set, num_negatives: int,
               model_scores: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Sample negatives using the configured mix of strategies.
        
        Args:
            user_positives: Set of item IDs the user has interacted with
            num_negatives: Total number of negatives to sample
            model_scores: Optional array of model scores for hard-negative mining
        
        Returns:
            Array of negative item IDs
        """
        # TODO: Implement multi-strategy sampling
        # 1. Calculate how many samples from each strategy based on ratios
        # 2. Sample from each strategy
        # 3. Concatenate and shuffle
        # 4. Update self.stats
        pass
    
    def get_stats(self) -> Dict[str, int]:
        """Return sampling statistics."""
        return dict(self.stats)


# Test your implementation:
# multi_sampler = MultiStrategyNegativeSampler(
#     NUM_ITEMS, item_counts,
#     strategy_ratios={'uniform': 0.3, 'popularity': 0.4, 'hard': 0.3}
# )
# test_positives = user_items[0]
# fake_scores = np.random.randn(NUM_ITEMS)
# negatives = multi_sampler.sample(test_positives, 100, model_scores=fake_scores)
# print(f"Sampled {len(negatives)} negatives")
# print(f"Stats: {multi_sampler.get_stats()}")

## 🏋️ Exercise 2: Implement Log-Q Correction for In-Batch Negatives

The in-batch negative sampling approach introduces a sampling bias because popular items appear more often as negatives. Implement the log-Q correction from Yi et al. (2019, Google) and demonstrate the difference.

In [ ]:
# 🏋️ Exercise 2: Log-Q correction

def compute_in_batch_loss_with_correction(
    user_embs: torch.Tensor,
    item_embs: torch.Tensor,
    item_ids: torch.Tensor,
    item_frequencies: torch.Tensor,
    temperature: float = 0.1,
    apply_correction: bool = True
) -> torch.Tensor:
    """
    Compute in-batch softmax cross-entropy with optional log-Q correction.
    
    Args:
        user_embs: (B, D) user embeddings
        item_embs: (B, D) item embeddings
        item_ids: (B,) item IDs for this batch
        item_frequencies: (num_items,) frequency of each item
        temperature: softmax temperature
        apply_correction: whether to apply log-Q correction
    
    Returns:
        Scalar loss
    """
    # TODO: Implement in-batch softmax loss
    # 1. Compute logits = user_embs @ item_embs.T / temperature
    # 2. If apply_correction, compute log(Q(item)) for each item in batch
    #    where Q(item) = item_freq / sum(item_freq)
    # 3. Subtract log(Q) from logits
    # 4. Labels are the diagonal (index i should match index i)
    # 5. Return cross-entropy loss
    pass


# Test:
# B, D = 128, 32
# user_e = F.normalize(torch.randn(B, D), dim=1)
# item_e = F.normalize(torch.randn(B, D), dim=1)
# ids = torch.randint(0, 1000, (B,))
# freqs = torch.ones(1000)  # uniform
# loss_no = compute_in_batch_loss_with_correction(user_e, item_e, ids, freqs, apply_correction=False)
# loss_yes = compute_in_batch_loss_with_correction(user_e, item_e, ids, freqs, apply_correction=True)
# print(f"Without correction: {loss_no.item():.4f}")
# print(f"With correction: {loss_yes.item():.4f}")

## 🏋️ Exercise 3: Build a Streaming Data Quality Monitor

Implement a `StreamingQualityMonitor` that processes incoming interaction batches and:
1. Tracks per-user interaction rates using an exponential moving average
2. Flags users whose rate exceeds a threshold as potential bots
3. Monitors the overall CTR and raises an alert if it deviates significantly

In [ ]:
# 🏋️ Exercise 3: Streaming quality monitor

class StreamingQualityMonitor:
    def __init__(self, rate_threshold: float = 500.0,
                 ctr_alert_zscore: float = 3.0,
                 ema_alpha: float = 0.1):
        """
        Args:
            rate_threshold: Max allowed interactions per hour per user
            ctr_alert_zscore: Z-score threshold for CTR anomaly detection
            ema_alpha: Exponential moving average smoothing factor
        """
        self.rate_threshold = rate_threshold
        self.ctr_alert_zscore = ctr_alert_zscore
        self.ema_alpha = ema_alpha
        
        # TODO: Initialize tracking state
        # - Per-user interaction counts and timestamps
        # - Running CTR statistics (mean, variance)
        # - Alerts list
        pass
    
    def process_batch(self, batch: Dict) -> Dict:
        """
        Process an incoming batch of interactions.
        
        Args:
            batch: Dict with 'user_ids', 'item_ids', 'labels', 'timestamp'
        
        Returns:
            Dict with 'flagged_users', 'ctr_alert', 'batch_ctr'
        """
        # TODO: Implement
        # 1. Update per-user interaction counts
        # 2. Check for rate violations
        # 3. Compute batch CTR and update running statistics
        # 4. Check for CTR anomalies
        pass


# Test:
# monitor = StreamingQualityMonitor()
# stream = StreamingDataGenerator(1000, 5000, item_popularity[:5000] / item_popularity[:5000].sum())
# for _ in range(20):
#     batch = stream.generate_batch(256)
#     result = monitor.process_batch(batch)
#     if result and result.get('flagged_users'):
#         print(f"Flagged users: {result['flagged_users']}")

## Summary

In this notebook, we covered the core data pipeline components for large-scale recommendation system training:

| Topic | Key Takeaway |
|-------|--------------|
| **Sampling Strategies** | Popularity-weighted sampling (α=0.75) offers the best trade-off for most use cases |
| **In-Batch Negatives** | Efficient but requires log-Q correction to avoid popularity bias |
| **Cached Negatives** | Good middle ground between freshness and negative pool size |
| **Data Formats** | Columnar formats (Parquet) are best for feature engineering; binary for streaming |
| **Streaming Pipelines** | Must handle label delay, feature freshness, and out-of-order events |
| **Data Quality** | Bot detection and deduplication are essential pre-training steps |

### Key References
- Mikolov et al., "Distributed Representations of Words and Phrases" (2013, Google)
- Yi et al., "Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations" (2019, Google)
- Rendle et al., "Neural Collaborative Filtering vs. Matrix Factorization Revisited" (2020, Google)
- Lindgren et al., "Efficient Training on Very Large Corpora via Gramian Estimation" (2021)

### Next Steps
In the next notebook (9.2), we will explore distributed training strategies for recommendation models, including parameter server architectures and embedding table sharding.